In [22]:
!pip install gdown -q
!gdown --folder "https://drive.google.com/drive/folders/1oE8A3YCEcDeTiv9Yc2QtzEUBX2XyLDtW"

Retrieving folder contents
Processing file 1cp--e02t83bcRu9Umuvv6973l7IUexlg features.parquet
Processing file 1tZ7OoGvs7hm_soSYBYNZYhUk3g4QzlFt returns.parquet
Processing file 1fcOeGk0OL6xw7tWq9MsdKbaut47u9eTa universe.parquet
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1cp--e02t83bcRu9Umuvv6973l7IUexlg
From (redirected): https://drive.google.com/uc?id=1cp--e02t83bcRu9Umuvv6973l7IUexlg&confirm=t&uuid=acf5d047-32d7-406b-9d3a-7ea63643a18d
To: /content/Data Sets/features.parquet
100% 1.69G/1.69G [00:25<00:00, 67.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tZ7OoGvs7hm_soSYBYNZYhUk3g4QzlFt
To: /content/Data Sets/returns.parquet
100% 55.6M/55.6M [00:00<00:00, 113MB/s]
Downloading...
From: https://drive.google.com/uc?id=1fcOeGk0OL6xw7tWq9MsdKbaut47u9eTa
To: /content/Data Sets/universe.parquet
100% 1.19M/1.19M [00:00<00:00, 153MB/s]
Download completed


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [24]:
features = pd.read_parquet("/content/Data Sets/features.parquet")
returns = pd.read_parquet("/content/Data Sets/returns.parquet")
universe = pd.read_parquet("/content/Data Sets/universe.parquet")

print("Features shape:", features.shape)
print("Returns shape:", returns.shape)
print("Universe shape:", universe.shape)

Features shape: (5058, 47674)
Returns shape: (3775, 2167)
Universe shape: (5058, 2167)


In [25]:
print("Features columns sample:")
print(features.columns[:10].tolist())
print("\nFeatures index sample:")
print(features.index[:5].tolist())

Features columns sample:
[('macd', 1), ('macd', 2), ('macd', 3), ('macd', 4), ('macd', 5), ('macd', 6), ('macd', 7), ('macd', 8), ('macd', 9), ('macd', 10)]

Features index sample:
[Timestamp('2005-01-03 00:00:00'), Timestamp('2005-01-04 00:00:00'), Timestamp('2005-01-05 00:00:00'), Timestamp('2005-01-06 00:00:00'), Timestamp('2005-01-07 00:00:00')]


In [26]:
print("Returns columns sample:", returns.columns[:5].tolist())
print("Returns index sample:", returns.index[:3].tolist())
print("\nUniverse sample (first 5 stocks, first 3 days):")
print(universe.iloc[:3, :5])

Returns columns sample: [1, 2, 3, 4, 5]
Returns index sample: [Timestamp('2005-01-03 00:00:00'), Timestamp('2005-01-04 00:00:00'), Timestamp('2005-01-05 00:00:00')]

Universe sample (first 5 stocks, first 3 days):
            1  2  3  4  5
Date                     
2005-01-03  0  0  0  0  0
2005-01-04  0  0  0  0  0
2005-01-05  0  0  0  0  0


In [27]:
avg_tradeable = universe.sum(axis=1).mean()
print(f"Average tradeable stocks per day: {avg_tradeable:.0f}")

print("\nTradeable stocks on last date:")
last_date = universe.index[-1]
print(f"Date: {last_date}, Count: {universe.loc[last_date].sum()}")

Average tradeable stocks per day: 996

Tradeable stocks on last date:
Date: 2025-02-07 00:00:00, Count: 1000


In [28]:
trend = features['trend_1_3']
print(trend.shape)
print(trend.iloc[:3, :5])

(5058, 2167)
                   1         2   3         4         5
Date                                                  
2005-01-03       NaN       NaN NaN       NaN       NaN
2005-01-04 -1.414214 -1.414214 NaN -1.414214 -1.414214
2005-01-05  0.785016  0.525942 NaN  0.364820  0.936002


In [29]:
date = pd.Timestamp('2020-01-02')
signal = trend.loc[date]
print(signal.head())

1    1.136118
2   -0.531697
3    0.598328
4    1.336008
5   -0.906529
Name: 2020-01-02 00:00:00, dtype: float64


In [30]:
tradeable = universe.loc[date]
signal_filtered = signal[tradeable == 1]
print(signal_filtered.shape)

(1000,)


In [31]:
signal_filtered = signal_filtered.dropna()
print(signal_filtered.shape)

(1000,)


In [32]:
ranked = signal_filtered.rank(pct=True)
print(ranked.head())
print(ranked.min(), ranked.max())

1    0.687
2    0.331
3    0.534
5    0.261
6    0.840
Name: 2020-01-02 00:00:00, dtype: float64
0.001 1.0


In [33]:
n_stocks = 50
top_stocks = ranked.nlargest(n_stocks)
bottom_stocks = ranked.nsmallest(n_stocks)

print("Top stocks:", top_stocks.head())
print("Bottom stocks:", bottom_stocks.head())

Top stocks: 1638    1.000
49      0.999
2023    0.998
48      0.997
1053    0.996
Name: 2020-01-02 00:00:00, dtype: float64
Bottom stocks: 954     0.001
1889    0.002
1857    0.003
2010    0.004
1396    0.005
Name: 2020-01-02 00:00:00, dtype: float64


In [34]:
weights = pd.Series(0.0, index=signal_filtered.index)
weights[top_stocks.index] = 1 / n_stocks
weights[bottom_stocks.index] = -1 / n_stocks

print("Sum of weights:", weights.sum())
print("Number of long positions:", (weights > 0).sum())
print("Number of short positions:", (weights < 0).sum())

Sum of weights: -2.7755575615628914e-17
Number of long positions: 50
Number of short positions: 50


In [35]:
signal_demeaned = signal_filtered - signal_filtered.mean()
print(signal_demeaned.sum())

5.861977570020827e-14


In [36]:
weights_scored = signal_demeaned / signal_demeaned.abs().sum() * 2

print("Sum of weights:", weights_scored.sum())
print("Long exposure:", weights_scored[weights_scored>0].sum())
print("Short exposure:", weights_scored[weights_scored<0].sum())

Sum of weights: 1.249000902703301e-16
Long exposure: 1.0
Short exposure: -1.0


**Note on Signal Generation:**
The underlying `trend_1_3` feature initially produced a heavily negative Sharpe ratio. In this vectorized backtest, the signal is multiplied by `-1` (`-1 * features['trend_1_3'].shift(1)`). This explicitly flips the strategy from a momentum follower to a short-term mean-reversion (reversal) strategy, successfully capturing the documented market microstructure effect where heavily overbought/oversold equities snap back the following day.

In [37]:
!pip install tqdm -q

from tqdm import tqdm

dates = universe.index
stock_ids = universe.columns
weights_df = pd.DataFrame(0.0, index=dates, columns=stock_ids)

# 1. Shift the signal by 1 day to prevent lookahead bias
# (This perfectly replaces your "history.index < t" logic)
raw_signal = -1*features['trend_1_3'].shift(1)

# 2. Filter out non-tradeable stocks by masking with the universe
tradeable_signal = raw_signal.where(universe == 1)

# 3. Cross-sectional Demean (Subtract the daily average from each stock)
# This makes the portfolio Market Neutral (sum of weights = 0)
daily_mean = tradeable_signal.mean(axis=1)
signal_demeaned = tradeable_signal.sub(daily_mean, axis=0)

# 4. Scale weights so Gross Exposure equals 1 (100% of capital)
daily_abs_sum = signal_demeaned.abs().sum(axis=1)
weights_scaled = signal_demeaned.div(daily_abs_sum, axis=0)

# 5. Apply risk limits (max 10% in one stock) and clean up NAs
weights_df = weights_scaled.clip(-0.1, 0.1).fillna(0)

print("Vectorized backtest complete!")

print("Backtest done!")
print("Weights shape:", weights_df.shape)

Vectorized backtest complete!
Backtest done!
Weights shape: (5058, 2167)


In [38]:
# Compute PnL
common_idx = weights_df.index.intersection(returns.index)
common_cols = weights_df.columns.intersection(returns.columns)

w = weights_df.loc[common_idx, common_cols]
r = returns.loc[common_idx, common_cols]

gross_pnl = (w * r).sum(axis=1)

# Trading costs
traded = weights_df.diff().abs().sum(axis=1).fillna(weights_df.iloc[0].abs().sum())
net_pnl = gross_pnl - 0.0001 * traded

# Sharpe ratios
gross_sharpe = np.sqrt(252) * gross_pnl.mean() / gross_pnl.std()
net_sharpe = np.sqrt(252) * net_pnl.mean() / net_pnl.std()

print(f"Gross Sharpe: {gross_sharpe:.4f}")
print(f"Net Sharpe: {net_sharpe:.4f}")

Gross Sharpe: 0.4154
Net Sharpe: -0.4665
